# Day 2 - Multimodal projector

- status: implemented_toy_not_executed_real_model
- stage: VLM_DAY_2
- paper_ids: llava_2023, qwen3_vl_2025
- dataset_ids: synthetic_toy, user_selected_nas_data
- seed: 42
- scope: educational implementation; real inference/training is opt-in

이 노트북은 다른 사용자 파일, 공유 환경, checkpoint를 자동으로 변경하지 않는다. 실제 데이터는
`/nas/datahub/min` 아래 사용자가 지정한 경로만 읽는다.

## 1. Learning question

vision encoder의 hidden dimension과 LLM hidden dimension이 다를 때 projector는 무엇을 학습하는가? spatial merge와 MLP projection을 분리한다.

## 2. Background theory

projector는 vision token을 LLM이 소비할 embedding space로 보낸다. 단순 linear, 2-layer MLP, resampler/Q-Former 등 설계가 있다. Qwen3-VL 계열은 spatial merge로 인접 token을 묶은 뒤 merger MLP로 LLM dimension에 맞춘다.

## 3. Paper connection

LLaVA류는 vision encoder와 LLM 사이 connector의 중요성을 보여준다. Qwen3-VL config는 patch size 16, spatial merge size 2와 vision-to-text output dimension을 명시한다. toy MLP는 교육용 근사다.

## 4. Input/output and shapes

merge 전 `[Gh,Gw,Dv]`, 2x2 merge 후 `[Gh/2,Gw/2,4*Dv]`, MLP 후 `[N/4,Dllm]`이다. merge는 token 수를 줄이는 대신 한 token이 더 넓은 영역을 대표한다.

In [ ]:
from pathlib import Path
import sys
import numpy as np

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "pyproject.toml").is_file()),
    Path("/nas/home/mhlee/vlm-foundation-7days"),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("project:", PROJECT_ROOT)
print("numpy:", np.__version__)

## 5. Minimal implementation

2x2 vision token grid를 channel 방향으로 합친 뒤 두 층 MLP로 projection한다.

In [ ]:
from vlm_foundation.projector import merge_spatial_tokens, mlp_project

rng = np.random.default_rng(42)
vision_grid = rng.normal(size=(4, 6, 3))
merged = merge_spatial_tokens(vision_grid, merge_size=2)
w1, b1 = rng.normal(size=(12, 8)), np.zeros(8)
w2, b2 = rng.normal(size=(8, 5)), np.zeros(5)
projected = mlp_project(merged, w1, b1, w2, b2)
print("vision grid:", vision_grid.shape)
print("merged:", merged.shape)
print("LLM-space tokens:", projected.reshape(-1, 5).shape)

## 6. Visualization sanity check

합쳐진 첫 token이 원래 grid의 어느 2x2 위치에서 왔는지 직접 확인한다.

In [ ]:
print("source top-left 2x2:\n", vision_grid[:2, :2])
print("merged first token:\n", merged[0, 0])

## 7. Experiment

merge size 1과 2에서 LLM visual token 수, projector input dimension을 비교한다.

In [ ]:
for merge_size in [1, 2]:
    output = merge_spatial_tokens(vision_grid, merge_size)
    print(f"merge={merge_size}: grid={output.shape[:2]}, input_dim={output.shape[-1]}")

## 8. Metrics

shape, trainable parameter 수, alignment loss, downstream task quality를 구분한다. projector loss 감소만으로 grounding이 좋아졌다고 말할 수 없다.

## 9. Interpretation

projector는 단순 dimension adapter이면서 modality alignment의 병목이다. 너무 강한 compression은 작은 글자와 위치 정보를 잃을 수 있다.

## 10. Failure cases

token order 불일치, vision/LLM dtype mismatch, 잘못된 merge reshape, frozen connector, special image token 개수 불일치를 확인한다.

## 11. Real-service implications

merge는 context와 KV cache를 줄이지만 detail을 희생한다. 문서 OCR과 장면 요약은 최적 token budget이 다르다.

## 12. Review questions

1. projector가 필요한 두 dimension은 무엇인가?
2. 2x2 merge가 token 수를 얼마나 줄이는가?
3. concatenate와 average merge의 정보 차이는?
4. projector만 학습하는 phase의 장단점은?
5. fine detail 작업에서 compression이 만드는 실패는?